# FinTech Loan Approval Prediction System

**Objective:** Determine loan eligibility based on applicant data using AutoML, Explainability (SHAP), and MLOps best practices.

### System Architecture
```
Raw Data (CSV)
    |
    v
Data Preprocessing (Imputation + Feature Engineering)
    |
    v
One-Hot Encoding (Categorical -> Numerical)
    |
    v
AutoML Model Comparison (LR, DT, RF, GB)
    |
    v
Best Model Selection (Highest Accuracy)
    |
    v
SHAP Explainability (Why approved/rejected?)
    |
    v
3D Visualization (Interactive Plotly)
    |
    v
MLOps: Model Export (.pkl) + CI/CD (GitHub Actions)
    |
    v
Live Prediction Demo
```

## Step 0: Install & Import Libraries

In [ ]:
!pip install pandas numpy scikit-learn shap plotly joblib matplotlib -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import shap
import plotly.express as px
import joblib
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

---
## Step 1: Data Loading & Exploration

In [ ]:
# Load the Loan Prediction dataset
url = "https://raw.githubusercontent.com/dphi-official/Datasets/master/Loan_Data/loan_train.csv"
df = pd.read_csv(url)

# Drop the unnamed index column if present
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
# Check missing values
print("Missing Values:")
print(df.isnull().sum())
print(f"\nDataset Info:")
df.info()

---
## Step 2: Data Preprocessing & Imputation

**Strategy:**
- Numerical columns -> fill with **median**
- Categorical columns -> fill with **mode**

In [ ]:
# Drop ID columns that are not useful for prediction
if 'Loan_ID' in df.columns:
    df = df.drop('Loan_ID', axis=1)

# Identify numerical and categorical columns
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove target from lists if present
if 'Loan_Status' in numerical_cols:
    numerical_cols.remove('Loan_Status')
if 'Loan_Status' in categorical_cols:
    categorical_cols.remove('Loan_Status')

print(f"Numerical columns: {numerical_cols}")
print(f"Categorical columns: {categorical_cols}")

# Imputation: Median for numerical, Mode for categorical
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(f"\nMissing values after imputation: {df.isnull().sum().sum()}")

---
## Step 3: Advanced Feature Engineering

Creating three new features:
- **TotalIncome** = ApplicantIncome + CoapplicantIncome
- **EMI** = LoanAmount / Loan_Amount_Term
- **BalanceIncome** = TotalIncome - EMI

In [ ]:
# Feature Engineering
df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['EMI'] = df['LoanAmount'] / df['Loan_Amount_Term']
df['BalanceIncome'] = df['TotalIncome'] - df['EMI']

print("Engineered Features Created:")
print(df[['TotalIncome', 'EMI', 'BalanceIncome']].describe().round(2))

---
## Step 4: One-Hot Encoding

Convert all categorical text columns into numerical format (1s and 0s).

In [ ]:
# One-Hot Encoding for categorical columns
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"Shape after encoding: {df_encoded.shape}")
print(f"\nEncoded columns: {list(df_encoded.columns)}")
df_encoded.head()

---
## Step 5: AutoML -- Model Comparison & Selection

Comparing 4 models and automatically selecting the best one:
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting

In [ ]:
# Split features (X) and target (y)
X = df_encoded.drop('Loan_Status', axis=1)
y = df_encoded['Loan_Status']

# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

In [ ]:
# Define the 4 models for AutoML comparison
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# Train and evaluate each model
results = {}
trained_models = {}

print("=" * 50)
print("       AutoML Model Comparison Results")
print("=" * 50)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    trained_models[name] = model
    print(f"{name:25s} -> Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Automatically select the best model
best_model_name = max(results, key=results.get)
best_model = trained_models[best_model_name]
best_accuracy = results[best_model_name]

print("=" * 50)
print(f"Best Model: {best_model_name} (Accuracy: {best_accuracy*100:.2f}%)")
print("=" * 50)

In [ ]:
# Detailed classification report for the best model
y_pred_best = best_model.predict(X_test)
print(f"Classification Report for {best_model_name}:")
print(classification_report(y_test, y_pred_best, target_names=['Rejected (0)', 'Approved (1)']))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

---
## Step 6: Explainability with SHAP

**SHAP (SHapley Additive exPlanations)** tells us *why* the model makes each decision.

- Features pushing towards **approval** appear in red (right)
- Features pushing towards **rejection** appear in blue (left)
- The wider the spread, the more important that feature is

In [ ]:
# Use SHAP to explain the best model
# Use TreeExplainer for tree-based models, otherwise LinearExplainer
if best_model_name in ['Decision Tree', 'Random Forest', 'Gradient Boosting']:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test)

    # For tree models, shap_values may be a list [class_0, class_1]
    if isinstance(shap_values, list):
        shap_vals_to_plot = shap_values[1]  # class 1 = Approved
    else:
        shap_vals_to_plot = shap_values
else:
    explainer = shap.LinearExplainer(best_model, X_train)
    shap_values = explainer.shap_values(X_test)
    shap_vals_to_plot = shap_values

print(f"SHAP values computed for {best_model_name}")

In [ ]:
# SHAP Summary Plot -- shows which features most impact the loan decision
print(f"SHAP Summary Plot for {best_model_name}:")
print("Red = pushes toward APPROVAL | Blue = pushes toward REJECTION")
print()
shap.summary_plot(shap_vals_to_plot, X_test, show=True)

### SHAP Explanation

The SHAP summary plot above reveals:
- **Credit_History** is typically the most influential feature -- having a credit history (value=1) strongly pushes the model toward approval.
- **TotalIncome** and **BalanceIncome** are important -- higher income increases approval chances.
- **LoanAmount** -- very high loan amounts can push toward rejection.
- **Property_Area** and **Married** status also contribute to the decision.

**Why does the model approve?** Primarily because the applicant has a positive credit history, adequate income relative to the loan, and a manageable EMI.

**Why does the model reject?** Usually due to no credit history, high loan-to-income ratio, or very high EMI relative to income.

---
## Step 7: Interactive 3D Visualization

3D Scatter Plot mapping:
- **X-axis:** TotalIncome
- **Y-axis:** EMI
- **Z-axis:** LoanAmount
- **Color:** Loan Status (Green = Approved, Red = Rejected)

In [ ]:
# Prepare data for 3D plot
plot_df = df.copy()
plot_df['Loan_Status_Label'] = plot_df['Loan_Status'].map({1: 'Approved', 0: 'Rejected'})

# Interactive 3D Scatter Plot
fig = px.scatter_3d(
    plot_df,
    x='TotalIncome',
    y='EMI',
    z='LoanAmount',
    color='Loan_Status_Label',
    color_discrete_map={'Approved': 'green', 'Rejected': 'red'},
    title='3D Visualization: Loan Approval by Income, EMI & Loan Amount',
    labels={
        'TotalIncome': 'Total Income (Rs)',
        'EMI': 'EMI (Monthly Payment)',
        'LoanAmount': 'Loan Amount (Rs)',
        'Loan_Status_Label': 'Loan Status'
    },
    opacity=0.7,
    width=900,
    height=700
)

fig.update_layout(
    scene=dict(
        xaxis_title='Total Income',
        yaxis_title='EMI',
        zaxis_title='Loan Amount'
    ),
    legend_title='Loan Status'
)

fig.show()

---
## Step 8: MLOps -- Model Export with joblib

Simulating a CI/CD pipeline by exporting the trained model as a `.pkl` file for production deployment.

In [ ]:
# Export the best model as a .pkl file
model_filename = 'best_model.pkl'
joblib.dump(best_model, model_filename)

# Verify the export
loaded_model = joblib.load(model_filename)
verify_pred = loaded_model.predict(X_test)
verify_accuracy = accuracy_score(y_test, verify_pred)

print(f"Model saved as: {model_filename}")
print(f"Model type: {best_model_name}")
print(f"Verification accuracy: {verify_accuracy*100:.2f}% (matches training: {verify_accuracy == best_accuracy})")
print("\nModel is ready for production deployment!")

### GitHub Actions CI/CD Configuration

The following YAML file should be placed at `.github/workflows/train.yml` in  repository.
we write this for automatically retrains the model whenever a new `data.csv` is pushed.

```yaml
name: Loan Model Training Pipeline

on:
  push:
    paths:
      - 'data.csv'

jobs:
  train:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout Repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'

      - name: Install Dependencies
        run: pip install pandas numpy scikit-learn shap joblib

      - name: Train Model
        run: python train.py

      - name: Upload Model Artifact
        uses: actions/upload-artifact@v4
        with:
          name: trained-model
          path: best_model.pkl
```

---
## Step 9: Live Prediction Demo

Testing the system with a new applicant to get a real-time APPROVED / REJECTED decision.

In [ ]:
# Create a new applicant
new_applicant = pd.DataFrame({
    'Gender': ['Male'],
    'Married': ['Yes'],
    'Dependents': ['2'],
    'Education': ['Graduate'],
    'Self_Employed': ['No'],
    'ApplicantIncome': [5000],
    'CoapplicantIncome': [2000],
    'LoanAmount': [150],
    'Loan_Amount_Term': [360],
    'Credit_History': [1.0],
    'Property_Area': ['Semiurban']
})

# Apply the same feature engineering
new_applicant['TotalIncome'] = new_applicant['ApplicantIncome'] + new_applicant['CoapplicantIncome']
new_applicant['EMI'] = new_applicant['LoanAmount'] / new_applicant['Loan_Amount_Term']
new_applicant['BalanceIncome'] = new_applicant['TotalIncome'] - new_applicant['EMI']

# Apply the same One-Hot Encoding
new_applicant_encoded = pd.get_dummies(new_applicant, columns=categorical_cols, drop_first=True)

# Align columns with training data (add missing columns as 0, remove extra columns)
for col in X.columns:
    if col not in new_applicant_encoded.columns:
        new_applicant_encoded[col] = 0
new_applicant_encoded = new_applicant_encoded[X.columns]

# Make prediction
prediction = best_model.predict(new_applicant_encoded)[0]

print("=" * 55)
print("         LOAN APPROVAL PREDICTION RESULT")
print("=" * 55)
print(f"  Applicant Income : Rs {new_applicant['ApplicantIncome'][0]:,}")
print(f"  Co-applicant Inc.: Rs {new_applicant['CoapplicantIncome'][0]:,}")
print(f"  Total Income     : Rs {new_applicant['TotalIncome'][0]:,}")
print(f"  Loan Amount      : Rs {new_applicant['LoanAmount'][0]:,}")
print(f"  EMI              : Rs {new_applicant['EMI'][0]:.2f}")
print(f"  Balance Income   : Rs {new_applicant['BalanceIncome'][0]:,.2f}")
print(f"  Credit History   : {'Yes' if new_applicant['Credit_History'][0] == 1 else 'No'}")
print("-" * 55)

if prediction == 1:
    print("  DECISION: *** APPROVED ***")
    print("  REASON: The applicant has a positive credit history,")
    print("  sufficient total income, and a manageable EMI-to-income")
    print("  ratio, making them a low-risk borrower.")
else:
    print("  DECISION: *** REJECTED ***")
    print("  REASON: The applicant's profile indicates higher risk")
    print("  due to factors such as insufficient income, high EMI")
    print("  relative to income, or lack of credit history.")
print("=" * 55)